<a href="https://colab.research.google.com/github/sanmeshh/pytorch_learning/blob/13.LSTM_on_toy_dataset/LSTM_13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install nltk

In [2]:
import nltk

In [3]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
import numpy as np
from collections import Counter
from nltk.tokenize import word_tokenize
import torch.optim as optim

In [5]:
document = """About the Program
What is the course fee for  Data Science Mentorship Program (DSMP 2023)
The course follows a monthly subscription model where you have to make monthly payments of Rs 799/month.
What is the total duration of the course?
The total duration of the course is 7 months. So the total course fee becomes 799*7 = Rs 5600(approx.)
What is the syllabus of the mentorship program?
We will be covering the following modules:
Python Fundamentals
Python libraries for Data Science
Data Analysis
SQL for Data Science
Maths for Machine Learning
ML Algorithms
Practical ML
MLOPs
Case studies
You can check the detailed syllabus here - https://learnwith.campusx.in/courses/CampusX-Data-Science-Mentorship-Program-637339afe4b0615a1bbed390
Will Deep Learning and NLP be a part of this program?
No, NLP and Deep Learning both are not a part of this program’s curriculum.
What if I miss a live session? Will I get a recording of the session?
Yes all our sessions are recorded, so even if you miss a session you can go back and watch the recording.
Where can I find the class schedule?
Checkout this google sheet to see month by month time table of the course - https://docs.google.com/spreadsheets/d/16OoTax_A6ORAeCg4emgexhqqPv3noQPYKU7RJ6ArOzk/edit?usp=sharing.
What is the time duration of all the live sessions?
Roughly, all the sessions last 2 hours.
What is the language spoken by the instructor during the sessions?
Hinglish
How will I be informed about the upcoming class?
You will get a mail from our side before every paid session once you become a paid user.
Can I do this course if I am from a non-tech background?
Yes, absolutely.
I am late, can I join the program in the middle?
Absolutely, you can join the program anytime.
If I join/pay in the middle, will I be able to see all the past lectures?
Yes, once you make the payment you will be able to see all the past content in your dashboard.
Where do I have to submit the task?
You don’t have to submit the task. We will provide you with the solutions, you have to self evaluate the task yourself.
Will we do case studies in the program?
Yes.
Where can we contact you?
You can mail us at nitish.campusx@gmail.com
Payment/Registration related questions
Where do we have to make our payments? Your YouTube channel or website?
You have to make all your monthly payments on our website. Here is the link for our website - https://learnwith.campusx.in/
Can we pay the entire amount of Rs 5600 all at once?
Unfortunately no, the program follows a monthly subscription model.
What is the validity of monthly subscription? Suppose if I pay on 15th Jan, then do I have to pay again on 1st Feb or 15th Feb
15th Feb. The validity period is 30 days from the day you make the payment. So essentially you can join anytime you don’t have to wait for a month to end.
What if I don’t like the course after making the payment. What is the refund policy?
You get a 7 days refund period from the day you have made the payment.
I am living outside India and I am not able to make the payment on the website, what should I do?
You have to contact us by sending a mail at nitish.campusx@gmail.com
Post registration queries
Till when can I view the paid videos on the website?
This one is tricky, so read carefully. You can watch the videos till your subscription is valid. Suppose you have purchased subscription on 21st Jan, you will be able to watch all the past paid sessions in the period of 21st Jan to 20th Feb. But after 21st Feb you will have to purchase the subscription again.
But once the course is over and you have paid us Rs 5600(or 7 installments of Rs 799) you will be able to watch the paid sessions till Aug 2024.
Why lifetime validity is not provided?
Because of the low course fee.
Where can I reach out in case of a doubt after the session?
You will have to fill a google form provided in your dashboard and our team will contact you for a 1 on 1 doubt clearance session
If I join the program late, can I still ask past week doubts?
Yes, just select past week doubt in the doubt clearance google form.
I am living outside India and I am not able to make the payment on the website, what should I do?
You have to contact us by sending a mail at nitish.campusx@gmai.com
Certificate and Placement Assistance related queries
What is the criteria to get the certificate?
There are 2 criterias:
You have to pay the entire fee of Rs 5600
You have to attempt all the course assessments.
I am joining late. How can I pay payment of the earlier months?
You will get a link to pay fee of earlier months in your dashboard once you pay for the current month.
I have read that Placement assistance is a part of this program. What comes under Placement assistance?
This is to clarify that Placement assistance does not mean Placement guarantee. So we dont guarantee you any jobs or for that matter even interview calls. So if you are planning to join this course just for placements, I am afraid you will be disappointed. Here is what comes under placement assistance
Portfolio Building sessions
Soft skill sessions
Sessions with industry mentors
Discussion on Job hunting strategies
"""

In [6]:
tokens=word_tokenize(document.lower())



In [7]:
vocab={'<UNK>':0}

for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token]=len(vocab)
vocab

{'<UNK>': 0,
 'about': 1,
 'the': 2,
 'program': 3,
 'what': 4,
 'is': 5,
 'course': 6,
 'fee': 7,
 'for': 8,
 'data': 9,
 'science': 10,
 'mentorship': 11,
 '(': 12,
 'dsmp': 13,
 '2023': 14,
 ')': 15,
 'follows': 16,
 'a': 17,
 'monthly': 18,
 'subscription': 19,
 'model': 20,
 'where': 21,
 'you': 22,
 'have': 23,
 'to': 24,
 'make': 25,
 'payments': 26,
 'of': 27,
 'rs': 28,
 '799/month': 29,
 '.': 30,
 'total': 31,
 'duration': 32,
 '?': 33,
 '7': 34,
 'months': 35,
 'so': 36,
 'becomes': 37,
 '799': 38,
 '*': 39,
 '=': 40,
 '5600': 41,
 'approx': 42,
 'syllabus': 43,
 'we': 44,
 'will': 45,
 'be': 46,
 'covering': 47,
 'following': 48,
 'modules': 49,
 ':': 50,
 'python': 51,
 'fundamentals': 52,
 'libraries': 53,
 'analysis': 54,
 'sql': 55,
 'maths': 56,
 'machine': 57,
 'learning': 58,
 'ml': 59,
 'algorithms': 60,
 'practical': 61,
 'mlops': 62,
 'case': 63,
 'studies': 64,
 'can': 65,
 'check': 66,
 'detailed': 67,
 'here': 68,
 '-': 69,
 'https': 70,
 '//learnwith.campusx.i

In [8]:
len(vocab)

289

In [9]:
#extract sentences from data
sentences=document.split('\n')

In [98]:
def text_to_indices(sentence,vocab):
  numerical_sent=[]

  for token in sentence:

    if token in vocab:
      numerical_sent.append(vocab[token])
    else:

      numerical_sent.append(vocab['<UNK>'])

  return numerical_sent

In [11]:
#lets tokenize our sentences
encoded_sentences=[]
for sent in sentences:
  encoded_sentences.append(text_to_indices(word_tokenize(sent.lower()),vocab))


about
the
program
what
is
the
course
fee
for
data
science
mentorship
program
(
dsmp
2023
)
the
course
follows
a
monthly
subscription
model
where
you
have
to
make
monthly
payments
of
rs
799/month
.
what
is
the
total
duration
of
the
course
?
the
total
duration
of
the
course
is
7
months
.
so
the
total
course
fee
becomes
799
*
7
=
rs
5600
(
approx
.
)
what
is
the
syllabus
of
the
mentorship
program
?
we
will
be
covering
the
following
modules
:
python
fundamentals
python
libraries
for
data
science
data
analysis
sql
for
data
science
maths
for
machine
learning
ml
algorithms
practical
ml
mlops
case
studies
you
can
check
the
detailed
syllabus
here
-
https
:
//learnwith.campusx.in/courses/campusx-data-science-mentorship-program-637339afe4b0615a1bbed390
will
deep
learning
and
nlp
be
a
part
of
this
program
?
no
,
nlp
and
deep
learning
both
are
not
a
part
of
this
program
’
s
curriculum
.
what
if
i
miss
a
live
session
?
will
i
get
a
recording
of
the
session
?
yes
all
our
sessions
are
recorded
,
so
ev

In [12]:
encoded_sentences

[[1, 2, 3],
 [4, 5, 2, 6, 7, 8, 9, 10, 11, 3, 12, 13, 14, 15],
 [2, 6, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 18, 26, 27, 28, 29, 30],
 [4, 5, 2, 31, 32, 27, 2, 6, 33],
 [2,
  31,
  32,
  27,
  2,
  6,
  5,
  34,
  35,
  30,
  36,
  2,
  31,
  6,
  7,
  37,
  38,
  39,
  34,
  40,
  28,
  41,
  12,
  42,
  30,
  15],
 [4, 5, 2, 43, 27, 2, 11, 3, 33],
 [44, 45, 46, 47, 2, 48, 49, 50],
 [51, 52],
 [51, 53, 8, 9, 10],
 [9, 54],
 [55, 8, 9, 10],
 [56, 8, 57, 58],
 [59, 60],
 [61, 59],
 [62],
 [63, 64],
 [22, 65, 66, 2, 67, 43, 68, 69, 70, 50, 71],
 [45, 72, 58, 73, 74, 46, 17, 75, 27, 76, 3, 33],
 [77, 78, 74, 73, 72, 58, 79, 80, 81, 17, 75, 27, 76, 3, 82, 83, 84, 30],
 [4, 85, 86, 87, 17, 88, 89, 33, 45, 86, 90, 17, 91, 27, 2, 89, 33],
 [92,
  93,
  94,
  95,
  80,
  96,
  78,
  36,
  97,
  85,
  22,
  87,
  17,
  89,
  22,
  65,
  98,
  99,
  73,
  100,
  2,
  91,
  30],
 [21, 65, 86, 101, 2, 102, 103, 33],
 [104,
  76,
  105,
  106,
  24,
  107,
  108,
  109,
  108,
  110,
  111,
  27,

In [13]:
train_seq=[]
#increasing the window size iteration by iteration
for sent in encoded_sentences:
  for i in range(1,len(sent)):
    train_seq.append(sent[:i+1])

In [14]:
len(train_seq)

942

In [15]:
train_seq[:5]

[[1, 2], [1, 2, 3], [4, 5], [4, 5, 2], [4, 5, 2, 6]]

In [16]:
overall=0
for seq in train_seq:
  overall=max(overall,len(seq))

print(overall)

62


In [17]:
#pre-padding
padded_seq=[]
for seq in train_seq:
  padded_seq.append([0]*(overall-len(seq))+seq)

In [20]:
padded_seq=torch.tensor(padded_seq,dtype=torch.long)

In [22]:
padded_seq.shape

torch.Size([942, 62])

In [23]:
X=padded_seq[:,:-1]
y=padded_seq[:,-1]

In [25]:
X.shape

torch.Size([942, 61])

In [26]:
class CustomDataset(Dataset):
  def  __init__(self,X,y):
    self.X=X
    self.y=y

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self,idx):
    return self.X[idx],self.y[idx]

In [27]:
dataset=CustomDataset(X,y)

In [28]:
dataloader=DataLoader(dataset,32,True)

In [29]:
for input,output in dataloader:
  print(input,output)

tensor([[  0,   0,   0,  ...,   2, 195,  22],
        [  0,   0,   0,  ...,  46, 124,   1],
        [  0,   0,   0,  ..., 243, 105, 240],
        ...,
        [  0,   0,   0,  ...,   8,  17, 108],
        [  0,   0,   0,  ..., 253, 266,  81],
        [  0,   0,   0,  ..., 181,  27,  28]]) tensor([ 25,   2,  30,  93,  81, 141, 213, 174, 235, 194,  86,  73,  15,  76,
        268,   5,  27,  27,  99,   2,  19,   6,   2,  36, 262,   8,  28, 215,
         24,  24, 267,  41])
tensor([[  0,   0,   0,  ...,  35, 142, 151],
        [  0,   0,   0,  ...,  24,  25,  94],
        [  0,   0,   0,  ..., 223, 190,  22],
        ...,
        [  0,   0,   0,  ...,   0, 123,  45],
        [  0,   0,   0,  ..., 132,  22, 179],
        [  0,   0,   0,  ..., 186,  78, 187]]) tensor([152,  26,  45, 259,  28, 253,  44,  94,  86,  93, 131,   2, 163, 149,
         41,  45, 106, 186,  22,  81,  24,  70,  78,  71,   3, 175,   2, 179,
        236,  86,   8, 135])
tensor([[  0,   0,   0,  ...,  85,  86, 155],
    

In [37]:
torch.rand(1,3,4)

tensor([[[0.6646, 0.3935, 0.3164, 0.0037],
         [0.6161, 0.5262, 0.6753, 0.7433],
         [0.2884, 0.9515, 0.1185, 0.3176]]])

In [58]:
class LSTMModel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size,100)
                      #embedding dimension=100,no.of neurons in NNs of each LSTm layer=150,keep batch_size as first dimension
    self.lstm=nn.LSTM(100,150,batch_first=True)
    self.fc=nn.Linear(150,vocab_size)

  def forward(self,x):
    embedded=self.embedding(x)
    intermediate_h_s,(final_h_s,final_c_s)=self.lstm(embedded)
    output=self.fc(final_h_s.squeeze(0))
    return output






In [59]:
model=LSTMModel(len(vocab))


In [60]:
device=torch.device("cuda" if torch.cuda.is_available() else 'cpu')

In [61]:
model.to(device)

LSTMModel(
  (embedding): Embedding(289, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (fc): Linear(in_features=150, out_features=289, bias=True)
)

In [62]:
epochs=50
lr=0.001
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=lr)

In [64]:
#training

for epoch in range(epochs):
  total_loss=0

  for batch_x,batch_y in dataloader:
    batch_x,batch_y=batch_x.to(device),batch_y.to(device)
    #reset gradients
    optimizer.zero_grad()

    #forward propagation
    output=model(batch_x)

    #loss calculation
    loss=criterion(output,batch_y)

    #back propagation
    loss.backward()

    #gradients_update
    optimizer.step()
    total_loss+=loss.item()

  print(f"Epoch:{epoch+1},Loss:{total_loss:.4f}")


Epoch:1,Loss:164.7634
Epoch:2,Loss:145.5689
Epoch:3,Loss:133.1040
Epoch:4,Loss:121.7029
Epoch:5,Loss:110.2826
Epoch:6,Loss:98.8495
Epoch:7,Loss:88.1094
Epoch:8,Loss:79.7358
Epoch:9,Loss:70.3881
Epoch:10,Loss:62.3941
Epoch:11,Loss:54.9835
Epoch:12,Loss:48.3304
Epoch:13,Loss:42.8507
Epoch:14,Loss:37.2976
Epoch:15,Loss:33.0999
Epoch:16,Loss:29.1369
Epoch:17,Loss:25.9629
Epoch:18,Loss:22.5065
Epoch:19,Loss:19.9609
Epoch:20,Loss:17.8373
Epoch:21,Loss:16.2534
Epoch:22,Loss:14.5765
Epoch:23,Loss:13.2737
Epoch:24,Loss:12.2198
Epoch:25,Loss:11.0277
Epoch:26,Loss:10.2949
Epoch:27,Loss:9.6612
Epoch:28,Loss:8.9025
Epoch:29,Loss:8.3815
Epoch:30,Loss:7.8997
Epoch:31,Loss:7.4295
Epoch:32,Loss:7.0957
Epoch:33,Loss:6.7648
Epoch:34,Loss:6.5671
Epoch:35,Loss:6.3256
Epoch:36,Loss:6.2399
Epoch:37,Loss:5.8879
Epoch:38,Loss:5.7458
Epoch:39,Loss:5.4780
Epoch:40,Loss:5.3730
Epoch:41,Loss:5.1576
Epoch:42,Loss:5.2004
Epoch:43,Loss:4.9693
Epoch:44,Loss:4.9552
Epoch:45,Loss:4.7888
Epoch:46,Loss:4.6319
Epoch:47,Los

In [99]:
#prediction
def prediction(model,vocab,text):
  tokenized=word_tokenize(text.lower())
  #text->numerical
  numerical_text=text_to_indices(tokenized,vocab)

  #paddding
  padded_text=torch.tensor([0]*(61-len(numerical_text))+numerical_text,dtype=torch.long).unsqueeze(0)

  #send to model
  output=model(padded_text)

  #predicted text

  value,index=torch.max(output,dim=1)
  index=index.item()
  res=list(vocab.keys())[index]

  #merge with text
  return (text+" "+res)







In [100]:
prediction(model,vocab,'The course follows a')

'The course follows a monthly'

In [103]:
tokens=10
import time
input_text='the course follows a monthly'

for i in range(tokens):
  output_text=prediction(model,vocab,input_text)
  print(output_text,sep=" ")
  input_text=output_text
  time.sleep(0.5)





the course follows a monthly subscription
the course follows a monthly subscription model
the course follows a monthly subscription model where
the course follows a monthly subscription model where you
the course follows a monthly subscription model where you have
the course follows a monthly subscription model where you have to
the course follows a monthly subscription model where you have to make
the course follows a monthly subscription model where you have to make monthly
the course follows a monthly subscription model where you have to make monthly payments
the course follows a monthly subscription model where you have to make monthly payments of


In [107]:
# Function to calculate accuracy
def calculate_accuracy(model, dataloader, device):
    model.eval()  # Set the model to evaluation mode
    correct = 0
    total = 0


    for batch_x, batch_y in dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        # Get model predictions
        outputs = model(batch_x)

        # Get the predicted word indices
        _, predicted = torch.max(outputs, dim=1)

        # Compare with actual labels
        correct += (predicted == batch_y).sum().item()
        total += batch_y.size(0)

    accuracy = correct / total * 100
    return accuracy

# Compute accuracy
accuracy = calculate_accuracy(model, dataloader, device)
print(f"Model Accuracy: {accuracy:.2f}%")

Model Accuracy: 95.44%
